In [1]:
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from pinecone import ServerlessSpec, Pinecone
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda,
)
import os
from sentence_transformers import CrossEncoder

d:\scaleRAG\productionRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\hp\AppData\Local\Temp\ipykernel_14728\35860561.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
load_dotenv()

True

In [3]:
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.7)

In [4]:
pc=Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

In [5]:
index_name = "prod-rag"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        serverless=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [6]:
file_path = "D:\ProdRAG\prodRAG\example.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()
print(type(loader))

<class 'langchain_community.document_loaders.pdf.PyPDFLoader'>


In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)

In [9]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

C:\Users\hp\AppData\Local\Temp\ipykernel_14728\2687298866.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1347.68it/s]


In [10]:
vectorstore = PineconeVectorStore.from_documents(texts, embedder, index_name=index_name)

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 10},)


In [12]:
reranker= CrossEncoder( "cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1479.50it/s]


In [13]:
def two_stage_retrieve(query, final_k=3):
    candidates = retriever.invoke(query)

    pairs = [(query, doc.page_content) for doc in candidates]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(candidates, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return [doc for doc, _ in ranked[:final_k]]

In [14]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [15]:
def chunks_docs(docs):
    return [doc.page_content for doc in docs]

In [16]:
rag_prompt = PromptTemplate.from_template(
    template="""You are a helpful assistant.
Use only the provided context to answer the question.
If the answer is not present in the context, say: "I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:"""
)

In [17]:
rag_chain = RunnableParallel(
    {
        "question": RunnablePassthrough(),
        "context": RunnableLambda(two_stage_retrieve)
                   | RunnableLambda(format_docs),
        
        
    }
)



In [18]:
result=rag_chain | rag_prompt | llm


In [19]:
op=result.invoke("What is the difference between module 4 and module 5?")

In [20]:
print(print(op.content))

Based on the provided context, Module 4: Smart Contracts and Ethereum deals with writing and deploying smart contracts using Solidity and Remix IDE. 

Module 5: Blockchain and AI Integration focuses on using blockchain to secure AI model sharing and training data provenance.

The main difference between Module 4 and Module 5 is that Module 4 is focused on smart contracts and Ethereum, while Module 5 is focused on integrating blockchain with AI, specifically securing AI model sharing and training data provenance.
None


In [26]:
questions = [
    "How does the proposed course integrate blockchain technology with Artificial Intelligence, Internet of Things, Cybersecurity, and Data Privacy, and why is this interdisciplinary approach important?",
    
    "Compare the consensus mechanisms Proof of Work (PoW), Proof of Stake (PoS), and Byzantine Fault Tolerance (BFT). In what scenarios would each mechanism be most suitable?",
    
    "Describe the complete workflow for developing and deploying a decentralized application (DApp) using the tools mentioned in the course proposal.",
    
    "Why has a Blockchain-based Secure Voting System been selected as the group project, and which blockchain concepts does it demonstrate?",
    
    "Explain the concept of AI model provenance. How can blockchain improve trust, transparency, and verification in AI systems?",
    
    "Discuss the hardware, software, and networking requirements of the course. Why are these resources necessary for blockchain development and AI integration?",
    
    "Analyze the course evaluation scheme. How does it ensure a balanced assessment of theoretical knowledge, practical implementation, and project-based learning?",
    
    "Explain how blockchain technology enhances security in IoT environments. Discuss its role in securing sensor data, communication networks, and edge devices.",
    
    "Evaluate the future scope of the proposed course. How can it contribute to research opportunities, industry collaborations, and institutional growth?",
    
    "Based on the departmental evaluation summary, propose improvements to the course that would satisfy the recommendations of all participating departments."
]

In [27]:
generated_answers=[]

In [28]:
generated_answers = []

for question in questions:
    op = result.invoke(question)
    generated_answers.append(op.content)

for answer in generated_answers:
    print(answer)
    print()

I don't know based on the provided context.

Based on the provided context, we can discuss the three consensus mechanisms: Proof of Work (PoW), Proof of Stake (PoS), and Byzantine Fault Tolerance (BFT).

**Proof of Work (PoW):**
PoW is a consensus mechanism that requires miners to solve complex mathematical puzzles to validate transactions and create new blocks. The miner who solves the puzzle first gets to add the block to the blockchain and is rewarded with newly minted cryptocurrency.

**Proof of Stake (PoS):**
PoS is a consensus mechanism that requires validators to "stake" their own cryptocurrency to participate in the validation process. The validator with the most staked cryptocurrency has a higher chance of being chosen to create the next block.

**Byzantine Fault Tolerance (BFT):**
BFT is a consensus mechanism that is based on the idea of achieving consensus in a network of nodes, even in the presence of faulty or malicious nodes. It works by having nodes agree on the state of

In [29]:
print(generated_answers)


["I don't know based on the provided context.", 'Based on the provided context, we can discuss the three consensus mechanisms: Proof of Work (PoW), Proof of Stake (PoS), and Byzantine Fault Tolerance (BFT).\n\n**Proof of Work (PoW):**\nPoW is a consensus mechanism that requires miners to solve complex mathematical puzzles to validate transactions and create new blocks. The miner who solves the puzzle first gets to add the block to the blockchain and is rewarded with newly minted cryptocurrency.\n\n**Proof of Stake (PoS):**\nPoS is a consensus mechanism that requires validators to "stake" their own cryptocurrency to participate in the validation process. The validator with the most staked cryptocurrency has a higher chance of being chosen to create the next block.\n\n**Byzantine Fault Tolerance (BFT):**\nBFT is a consensus mechanism that is based on the idea of achieving consensus in a network of nodes, even in the presence of faulty or malicious nodes. It works by having nodes agree on